# VAE vision-codec: load a checkpoint, run inference, compute metrics

This notebook is the end-to-end example for evaluating a trained Track A1 KL-VAE
from a notebook / Colab. Everything routes through the shared helpers in
`toylib_projects.wm.vision_encoder.inference` and `...metrics`, so the encode /
decode / normalization path is identical to the offline evaluator and the Stage 2
datagen pipeline.

**Flow**
1. Set up the environment (clone repo + install).
2. Point at a checkpoint directory and a compiled `vae_test.h5`.
3. Load frames + state, load the VAE, reconstruct.
4. Compute each metric directly (PSNR, SSIM, region-PSNR, KL-per-channel).
5. Run the full `evaluate_checkpoint` report (overall + stratified + baselines + stage gate).
6. Visualize input vs. reconstruction.

You supply the two paths in the **Config** cell — the checkpoint you downloaded
and a compiled test `.h5`. Nothing else needs editing.

## 1. Environment setup (Colab)

On Colab, clone the repo and install it. Locally (running with `uv`) you can
**skip this cell** — the package is already importable.

> Adjust `REPO_URL` / branch as needed. The `-e` install picks up the package so
> `from toylib_projects...` works.

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/anujkhare/toylib.git"  # <-- set me
    !git clone --depth 1 {REPO_URL} /tmp/toylib
    %cd /tmp/toylib
    # Install the project + its deps (jax, orbax, h5py, hdf5plugin, pandas, pillow).
    !pip install -q -e .
    # If your checkpoints live on Google Drive, mount it:
    # from google.colab import drive; drive.mount('/content/drive')
else:
    print("Not on Colab — assuming the toylib package is already importable.")

Cloning into '/tmp/toylib'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 147 (delta 14), reused 81 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 2.93 MiB | 8.73 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/tmp/toylib
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.6 MB/s eta 0:00:00
  Building editable for toylib (pyproject.toml) ... done


In [ ]:
#@title Auth GCS for checkpoint storage
from google.colab import auth
from google.colab import userdata
auth.authenticate_user()
print('User authenticated.')

project_id = userdata.get('GCS_PROJECT_ID')  # set in Colab's left sidebar > Secrets


In [2]:
#@title Clone data
!gcloud config set project {project_id}
!mkdir -p /tmp/data/
!gcloud storage cp --recursive gs://toylib-wm-data/data/ /tmp/data/

ERROR: (gcloud.storage.cp) You do not currently have an active account selected.
Please run:

  $ gcloud auth login

to obtain new credentials.

If you have already logged in with a different account, run:

  $ gcloud config set account ACCOUNT

to select an already authenticated account to use.


## 2. Config — set your paths

- `CHECKPOINT_DIR`: the orbax checkpoint folder `train` wrote (the `.../<run_id>`
  directory that contains numbered step subfolders). Local path or `gs://...`.
- `TEST_H5`: a compiled `vae_{val,test}.h5` (frames + `source/` state).
- `BASE_CH` / `LATENT_CHANNELS`: **must match the trained model** (defaults 64 / 4).

`CHECKPOINT_STEP=None` restores the latest saved step.

In [ ]:
CHECKPOINT_DIR = "gs://tinystories-checkpoints/wm-vae/20260702T002831/"   # <-- set me
CHECKPOINT_STEP = None                         # None = latest
TEST_H5 = "/tmp/data/compiled/vae_test.h5"        # <-- set me

BASE_CH = 64
LATENT_CHANNELS = 4

# Number of frames to pull for the quick interactive cells (3-4). The full
# evaluate_checkpoint call in section 5 can run over the whole set.
N_FRAMES = 256
BATCH_SIZE = 64

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from toylib_projects.wm.vision_encoder import inference
from toylib_projects.wm.vision_encoder import metrics
from toylib_projects.wm.vision_encoder import evaluate

## 3. Load frames + state, load the VAE, reconstruct

`load_frames` returns the uint8 frames, a dict of per-frame RAM state
(`ball_x`, `ball_y`, `paddle_x`, `mode`, `score`), and the `PreprocessConfig` the
dataset was compiled with (needed so `ram_to_pixel` lands on the right pixels).

In [ ]:
frames, source, config = inference.load_frames(TEST_H5, n=N_FRAMES)
print("frames:", frames.shape, frames.dtype)
print("state keys:", list(source.keys()))
print("preprocess config:", config)

vae = inference.load_vae(
    CHECKPOINT_DIR, CHECKPOINT_STEP,
    base_ch=BASE_CH, latent_channels=LATENT_CHANNELS,
)
print("loaded VAE at step:", inference.latest_step(CHECKPOINT_DIR) if CHECKPOINT_STEP is None else CHECKPOINT_STEP)

# Full round-trip decode(encode(frames)) -> uint8 frames.
recons = inference.reconstruct(vae, frames, batch_size=BATCH_SIZE)
print("recons:", recons.shape, recons.dtype)

## 4. Compute the metrics directly

Each metric is plain numpy in / numpy out. Per-frame variants return `(N,)` so you
can slice / histogram; the scalar wrappers just average.

In [ ]:
# --- Reconstruction fidelity ---
psnr_pf = metrics.psnr_per_frame(frames, recons)
ssim_pf = metrics.ssim_per_frame(frames, recons)
print(f"PSNR  mean={np.mean(psnr_pf[np.isfinite(psnr_pf)]):.3f} dB")
print(f"SSIM  mean={np.mean(ssim_pf):.4f}")

# --- Physics via known RAM state (no trained detector) ---
ball_pf = metrics.ball_region_psnr_per_frame(
    frames, recons, source["ball_x"], source["ball_y"], config, size=16,
)
paddle_pf = metrics.paddle_region_psnr_per_frame(
    frames, recons, source["paddle_x"], config, size=24,
)
# nan = ball off-screen (between lives); average over finite entries only.
print(f"ball_region_psnr    mean={np.nanmean(ball_pf[np.isfinite(ball_pf)]):.3f} dB "
      f"({np.isfinite(ball_pf).sum()}/{len(ball_pf)} frames with ball in play)")
print(f"paddle_region_psnr  mean={np.nanmean(paddle_pf[np.isfinite(paddle_pf)]):.3f} dB")

In [ ]:
# --- Latent diagnostics: KL budget per channel (dead-channel check) ---
mu, log_sigma_sq = inference.encode_latent_stats(vae, frames, batch_size=BATCH_SIZE)
kl_c = metrics.kl_per_channel(mu, log_sigma_sq)
print("KL per channel (nats):", np.round(kl_c, 4))
print(f"  min={kl_c.min():.4f}  mean={kl_c.mean():.4f}")
print("  (a channel near 0 is 'dead' / collapsed capacity)")

## 5. Full evaluation report

`evaluate_checkpoint` does everything above over a (larger) set and adds:
stratification by game mode and score bucket, naive baselines
(identity / mean-frame / bilinear-8×) for context, and a stage-gate verdict.

Set `max_frames=None` to run the whole test set. The heavy arrays are returned
under `results['arrays']` for further drill-down (they are not written to JSON).

In [ ]:
results = evaluate.evaluate_checkpoint(
    checkpoint_dir=CHECKPOINT_DIR,
    test_path=TEST_H5,
    step=CHECKPOINT_STEP,
    base_ch=BASE_CH,
    latent_channels=LATENT_CHANNELS,
    batch_size=BATCH_SIZE,
    max_frames=2000,   # None = whole test set
)
evaluate.print_report(results)

In [ ]:
# Drill into the stratified tables as DataFrames.
import pandas as pd

if results["by_mode"]:
    display(pd.DataFrame.from_dict(results["by_mode"], orient="index").round(3))
if results["by_score_bucket"]:
    display(pd.DataFrame.from_dict(results["by_score_bucket"], orient="index").round(3))

## 6. Visualize input vs. reconstruction

A qualitative check: top row = input, bottom row = VAE reconstruction. Look at
the ball (small, easy to blur away) and the paddle.

In [ ]:
def show_reconstructions(inputs, recons, n=8, start=0):
    n = min(n, len(inputs) - start)
    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4.5))
    for i in range(n):
        idx = start + i
        axes[0, i].imshow(inputs[idx]); axes[0, i].axis("off")
        axes[1, i].imshow(recons[idx]); axes[1, i].axis("off")
        axes[0, i].set_title(f"#{idx}", fontsize=8)
    axes[0, 0].set_ylabel("input", fontsize=10)
    axes[1, 0].set_ylabel("recon", fontsize=10)
    fig.tight_layout()
    plt.show()

show_reconstructions(frames, recons, n=8)

In [ ]:
# Histograms of the per-frame metrics from section 4.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].hist(psnr_pf[np.isfinite(psnr_pf)], bins=30); axes[0].set_title("PSNR (dB)")
axes[1].hist(ssim_pf, bins=30); axes[1].set_title("SSIM")
axes[2].hist(ball_pf[np.isfinite(ball_pf)], bins=30); axes[2].set_title("ball region PSNR (dB)")
fig.tight_layout()
plt.show()

## 7. (Optional) Save the JSON report

`_json_safe` drops the heavy arrays and coerces numpy → python so the report is
dumpable. Same content `main()` writes from the CLI.

In [ ]:
import json

with open("eval_report.json", "w") as f:
    json.dump(evaluate._json_safe(results), f, indent=2)
print("wrote eval_report.json")